# ECG pipeline modernization: a supported narrative walkthrough

This notebook is the repository's canonical explanation and orchestration layer. It presents a historical educational notebook project as a **modernization and reproducibility case study**: versioned provenance, explicit data contracts, subject-aware evaluation discipline, and auditable operational evidence. Implementation remains in `src/ecg_anomaly_detection/`, configuration, the `ecg-data` CLI, and tests.

> **Use boundary:** This repository is not medical software, clinical software, or production ML. It makes no clinical, diagnostic, deployment, or medical-utility claims. Validation evidence is not benchmark evidence.

## 1. Repository purpose and modernization context

The supported workflow replaces notebook-bound transformations with testable package boundaries. The notebook reads those contracts; it does not reimplement them. The diagram below is a modern repository-owned asset, not an image from the historical archive.

![Supported modern pipeline and evidence lineage](../reports/figures/modern-pipeline-lineage.svg)

In [1]:
from dataclasses import asdict
from pathlib import Path
import json

from ecg_anomaly_detection.config import RepositoryPaths, load_dataset_config
from ecg_anomaly_detection.labels import load_annotation_mapping
from ecg_anomaly_detection.windows import load_window_config
from ecg_anomaly_detection.splitting import load_split_config
from ecg_anomaly_detection.training import load_training_config
from ecg_anomaly_detection.evaluation import load_evaluation_config
from ecg_anomaly_detection.benchmark_policy import load_benchmark_policy

paths = RepositoryPaths.discover()
root = paths.root
{'repository_root': str(root), 'package_backed': True}

{'repository_root': '/Users/jaredgodar/Developer/portfolio/ecg_anomaly_detection',
 'package_backed': True}

## 2. Historical archive boundary

The notebooks under `archive/original_2022/` are preserved historical artifacts. They are unsupported, may contain stale errors and duplicated exploratory logic, and are not executed by this walkthrough. Their saved metrics remain historical results with known random beat-window split limitations; they do not demonstrate generalization to unseen patients. No historical image is reused here.

In [2]:
archive = root / 'archive' / 'original_2022'
{'archive_exists': archive.is_dir(), 'archive_role': 'preserved and unsupported'}

{'archive_exists': True, 'archive_role': 'preserved and unsupported'}

## 3. Configuration-driven workflow

Each stage validates a committed TOML contract before touching data. Loading these contracts is deterministic and data-independent.

In [3]:
dataset = load_dataset_config(paths.configs / 'mitdb-v1.0.0.toml')
mapping = load_annotation_mapping(paths.configs / 'annotation-map-v1.toml')
windowing = load_window_config(paths.configs / 'windowing-v1.toml')
splitting = load_split_config(paths.configs / 'splitting-v2.toml')
training = load_training_config(paths.configs / 'training-baseline-v1.toml')
evaluation = load_evaluation_config(paths.configs / 'evaluation-baseline-v1.toml')

{
    'dataset': f'{dataset.slug}@{dataset.version}',
    'records': len(dataset.record_ids),
    'mapping': f'{mapping.name}@{mapping.version}',
    'windowing': f'{windowing.name}@{windowing.version}',
    'splitting': f'{splitting.name}@{splitting.version}',
    'training': f'{training.name}@{training.version}',
    'evaluation': f'{evaluation.name}@{evaluation.version}',
}

{'dataset': 'mitdb@1.0.0',
 'records': 48,
 'mapping': 'historical-binary@1.0.0',
 'windowing': 'historical-six-second@1.0.0',
 'splitting': 'subject-aware-holdout@2.0.0',
 'training': 'seeded-random-projection-nearest-centroid@1.0.0',
 'evaluation': 'validation-baseline-metrics@1.0.0'}

## 4. Acquisition and source validation

The acquisition stage retrieves the configured public MIT-BIH release into the ignored raw-data zone, checks the repository-reviewed expected file inventory, and records acquisition and SHA-256 evidence. This notebook deliberately does not download the complete dataset during routine validation. A local full run uses the supported command shown later.

In [4]:
{
    'source': dataset.source_url,
    'sample_rate_hz': dataset.sample_rate_hz,
    'required_files': len(dataset.expected_files),
    'reviewed_source_digests': len(dataset.expected_source_files),
    'raw_data_committed': False,
}

{'source': 'https://physionet.org/content/mitdb/1.0.0/',
 'sample_rate_hz': 360,
 'required_files': 144,
 'reviewed_source_digests': 144,
 'raw_data_committed': False}

## 5. Annotation mapping

The package applies a closed-world, versioned mapping. Every source symbol must be assigned to a target or explicitly excluded; unknown symbols fail closed. Per-record JSON reports preserve inclusion, exclusion, and target counts.

In [5]:
{
    'unknown_symbol_policy': mapping.unknown_symbol_policy,
    'targets': {rule.name: list(rule.symbols) for rule in mapping.targets},
    'explicitly_excluded_symbols': list(mapping.excluded_symbols),
}

{'unknown_symbol_policy': 'error',
 'targets': {'reference_normal': ['N'],
  'selected_other': ['L',
   'R',
   'V',
   '/',
   'A',
   'f',
   'F',
   'j',
   'a',
   'E',
   'J',
   'e',
   'S']},
 'explicitly_excluded_symbols': ['[',
  '!',
  ']',
  'x',
  '(',
  ')',
  'p',
  't',
  'u',
  '`',
  "'",
  '^',
  '|',
  '~',
  '+',
  's',
  'T',
  '*',
  'D',
  '=',
  '"',
  '@',
  'Q',
  '?']}

## 6. Window extraction

The package converts configured time geometry into samples, selects the configured channel, excludes incomplete boundary windows, and retains record ID, annotation location, source symbol, and target lineage. It also reports adjacent overlap rather than hiding it.

In [6]:
{
    'pre_seconds': windowing.pre_seconds,
    'post_seconds': windowing.post_seconds,
    'channel_index': windowing.channel_index,
    'boundary_policy': windowing.boundary_policy,
    'lineage_fields': ['record_id', 'center_sample_index', 'source_symbol', 'target_value'],
}

{'pre_seconds': 3.0,
 'post_seconds': 3.0,
 'channel_index': 0,
 'boundary_policy': 'exclude',
 'lineage_fields': ['record_id',
  'center_sample_index',
  'source_symbol',
  'target_value']}

## 7. Subject-aware splitting and split diagnostics

Split schema v2 maps records to subjects and assigns complete subjects to deterministic train, validation, and protected test partitions. Package assertions reject subject or record crossover. A generated split-quality summary records disjointness, partition counts, class coverage, ratio deviations, and configured warning/failure thresholds.

In [7]:
{
    'schema_version': splitting.schema_version,
    'strategy': splitting.strategy,
    'seed': splitting.seed,
    'ratios': {
        'train': splitting.train_ratio,
        'validation': splitting.validation_ratio,
        'test': splitting.test_ratio,
    },
    'record_subject_mappings': len(splitting.record_subjects),
}

{'schema_version': 2,
 'strategy': 'seeded-subject-shuffle',
 'seed': 2022,
 'ratios': {'train': 0.7, 'validation': 0.15, 'test': 0.15},
 'record_subject_mappings': 48}

## 8. Baseline training and validation-only evaluation

Training resolves only training shards from the model-ready index. Evaluation then loads the frozen model and resolves only validation shards, verifying content digests before scoring. Validation metrics are pipeline evidence—not benchmark evidence, protected-test evidence, or proof of generalization.

In [8]:
{
    'estimator': training.estimator,
    'training_seed': training.seed,
    'training_partition': 'train',
    'evaluation_partition': evaluation.partition,
    'test_partition_accessed': False,
}

{'estimator': 'random-projection-nearest-centroid',
 'training_seed': 2022,
 'training_partition': 'train',
 'evaluation_partition': 'validation',
 'test_partition_accessed': False}

## 9. Reproducibility evidence, run manifests, and artifact lineage

A supported full run writes environment, runtime, resource, configuration, report, artifact, and digest evidence beneath an ignored UUID-scoped run directory. The final run manifest links those records. The next cell reads only existing JSON evidence; when local artifacts are absent, it returns an explicit status and continues cleanly.

In [9]:
run_root = paths.artifacts / 'runs'
run_manifests = sorted(run_root.glob('*/run-manifest.json')) if run_root.is_dir() else []

if run_manifests:
    latest_manifest_path = run_manifests[-1]
    latest_manifest = json.loads(latest_manifest_path.read_text(encoding='utf-8'))
    evidence_status = {
        'status': 'local evidence available',
        'manifest': str(latest_manifest_path.relative_to(root)),
        'schema_version': latest_manifest.get('schema_version'),
        'top_level_fields': sorted(latest_manifest),
    }
else:
    evidence_status = {
        'status': 'no local run evidence; execute the supported CLI when data access is intended',
        'expected_location': 'artifacts/runs/<run-id>/run-manifest.json',
    }

evidence_status

{'status': 'no local run evidence; execute the supported CLI when data access is intended',
 'expected_location': 'artifacts/runs/<run-id>/run-manifest.json'}

Run the complete supported workflow from the repository root when local data acquisition is intended:

```fish
uv run ecg-data run-pipeline \
  --repository-root . \
  --dataset-config configs/mitdb-v1.0.0.toml \
  --mapping-config configs/annotation-map-v1.toml \
  --window-config configs/windowing-v1.toml \
  --split-config configs/splitting-v2.toml \
  --training-config configs/training-baseline-v1.toml \
  --evaluation-config configs/evaluation-baseline-v1.toml
```

## 10. Benchmark governance boundary

The held-out test partition is intentionally unavailable for routine evaluation. The benchmark policy is fail-closed: test evaluation is disabled and any future access requires separate review, explicit opt-in, immutable lineage, disclosure, and archival controls. This notebook validates that policy without opening data.

In [10]:
benchmark_policy = load_benchmark_policy(paths.configs / 'benchmark-policy-v1.toml')
assert benchmark_policy.protected_partition == 'test'
assert benchmark_policy.test_evaluation_enabled is False
assert benchmark_policy.explicit_future_opt_in_required is True
{
    'policy': f'{benchmark_policy.policy_id}@{benchmark_policy.version}',
    'protected_partition': benchmark_policy.protected_partition,
    'test_evaluation_enabled': benchmark_policy.test_evaluation_enabled,
    'future_opt_in_required': benchmark_policy.explicit_future_opt_in_required,
}

{'policy': 'benchmark-governance-v1@1.0.0',
 'protected_partition': 'test',
 'test_evaluation_enabled': False,
 'future_opt_in_required': True}

## 11. Portfolio interpretation and limitations

The engineering result is a reproducible, configuration-driven local pipeline with explicit provenance, validation, lineage, subject-aware preparation, testable transformations, and operational evidence. It demonstrates data-engineering and platform-engineering judgment.

It does **not** establish model quality, unseen-population generalization, clinical validity, diagnostic value, medical utility, production readiness, or deployment fitness. Historical notebook metrics remain historical artifacts. Modern validation-only metrics remain development evidence. No protected test metric or supported benchmark is produced here.